# FIK FAIR v11: rank correction with feature residuals

The team reports public RMSE 2.1068098327 for v10. This experiment tests whether features explain departures from the ordered target trend, and averages a selected rank/residual estimator with v10 at fixed 50:50 weights. No leaderboard score is used in fitting.

Attach official competition files and the actual `submission_v10.csv`. Optionally attach `v10_oof.csv` to reproduce paired comparisons. CPU is sufficient. For a strict rerun, preserve file order and use the library versions saved in `requirements-v11.txt`. The organiser permission for order use is reported by the team.

In [ ]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.special import betainc
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold, RepeatedKFold

ON_KAGGLE = Path('/kaggle/input').exists()
# Set explicit paths here if multiple candidate inputs are attached.
DATA_DIR = None
V10_PATH = None
V10_OOF_PATH = None
if ON_KAGGLE:
    input_root = Path('/kaggle/input')
    if DATA_DIR is None:
        paths = []
        for path in input_root.rglob('train.csv'):
            columns = pd.read_csv(path, nrows=0).columns
            if {'record_id', 'target', 'feature_24'} <= set(columns) and (path.parent / 'test.csv').exists():
                paths.append(path.parent)
        if len(paths) != 1:
            raise ValueError(f'Set DATA_DIR explicitly; found {paths}')
        DATA_DIR = paths[0]
    if V10_PATH is None:
        paths = list(input_root.rglob('submission_v10.csv'))
        if len(paths) != 1:
            raise ValueError(f'Attach only the v10 submission, or set V10_PATH explicitly: {paths}')
        V10_PATH = paths[0]
    if V10_OOF_PATH is None:
        paths = list(input_root.rglob('v10_oof.csv'))
        if len(paths) == 1:
            V10_OOF_PATH = paths[0]
    OUT = Path('/kaggle/working')
else:
    root = Path.cwd()
    if not (root / 'data/train.csv').exists():
        root = root.parent
    DATA_DIR = Path(DATA_DIR) if DATA_DIR is not None else root / 'data'
    V10_PATH = Path(V10_PATH) if V10_PATH is not None else root / 'outputs/fikfair_v10/submission_v10.csv'
    V10_OOF_PATH = Path(V10_OOF_PATH) if V10_OOF_PATH is not None else root / 'outputs/v11_eda/v10_oof.csv'
    OUT = root / 'outputs/fikfair_v11'
OUT.mkdir(parents=True, exist_ok=True)
DATA_DIR, V10_PATH = Path(DATA_DIR), Path(V10_PATH)
print('Data:', DATA_DIR, '\nv10:', V10_PATH, '\nOutput:', OUT)

from sklearn.linear_model import Ridge
import scipy
import sklearn
import sys
import hashlib
import platform
VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'scipy': scipy.__version__, 'scikit-learn': sklearn.__version__}
print('Environment:', VERSIONS)


In [ ]:
tr = pd.read_csv(DATA_DIR / 'train.csv')
te = pd.read_csv(DATA_DIR / 'test.csv')
template = pd.read_csv(DATA_DIR / 'submission_template.csv')
FEATURES = [c for c in tr.columns if c.startswith('feature_')]
X, y, Xte = tr[FEATURES].to_numpy(), tr.target.to_numpy(), te[FEATURES].to_numpy()
assert len(FEATURES) == 24
assert np.isfinite(X).all() and np.isfinite(Xte).all() and np.isfinite(y).all()
assert tr.record_id.is_unique and te.record_id.is_unique
assert not set(tr.record_id) & set(te.record_id)
v10_file = pd.read_csv(V10_PATH)
assert list(v10_file.columns) == ['record_id', 'prediction']
assert len(v10_file) == len(te) and v10_file.record_id.is_unique
assert set(v10_file.record_id) == set(te.record_id)
v10_test = te[['record_id']].merge(v10_file, on='record_id', validate='one_to_one').prediction.to_numpy()
assert np.isfinite(v10_test).all()
# For this particular v10 run the chosen rule was mean(quantile, sorted).
# Checking known predictions prevents accidentally attaching another version.
expected_head = np.array([93.291060, 84.052009, 78.284577, 73.876091, 71.114928])
assert np.allclose(v10_test[:5], expected_head, atol=2e-5, rtol=0), 'This is not the v10 submission audited here'
baseline_oof = None
if V10_OOF_PATH is not None and Path(V10_OOF_PATH).exists():
    saved = pd.read_csv(V10_OOF_PATH)
    assert saved.record_id.is_unique and set(saved.record_id) == set(tr.record_id)
    saved = tr[['record_id']].merge(saved, on='record_id', validate='one_to_one')
    assert np.allclose(saved.target, y)
    baseline_oof = saved[['v10_rep1', 'v10_rep2', 'v10_rep3']].to_numpy().T
print('Train/test:', X.shape, Xte.shape, '| archived baseline OOF:', baseline_oof is not None)


## Candidate models

Nine candidates test PCA dimension averaging, subsampling, a Brownian-bridge rank prior, and regularized feature residuals. Residual models subtract smooth position trends from training features and targets, fit Ridge to the remaining variation, and predict deviations around the query rank trend. Everything is fitted within the relevant training partition.

The choice between candidates uses inner five-fold validation. The candidate catalogue was designed after exploring this dataset, so research-selection bias is not eliminated by nested evaluation.

In [ ]:
def rank(n):
    return (np.arange(n)+.5)/n

def curve(t, u, smooth=0):
    q=rank(len(t))
    ys=IsotonicRegression(increasing=False).fit_transform(q,t)
    if smooth:
        ys=gaussian_filter1d(ys, smooth*len(t),mode='nearest')
    return np.interp(u,q,ys)

def beta_curve(t, m):
    # Integrate the smoothed training rank curve against each query order
    # statistic's Beta(j, m+1-j) marginal. This is a modelling approximation.
    q=rank(len(t))
    ys=IsotonicRegression(increasing=False).fit_transform(q,t)
    edges=np.r_[0,(q[1:]+q[:-1])/2,1]
    j=np.arange(1,m+1)[:,None]
    w=np.diff(betainc(j,m+1-j,edges[None,:]),axis=1)
    return w@ys

def neighbor_warp(A,B,k=20,alpha=.5):
    # Fit feature transform and calibration on the current training partition.
    mu,sd=A.mean(0),A.std(0)
    pc=PCA(n_components=12).fit((A-mu)/sd)
    ZA=pc.transform((A-mu)/sd)/np.sqrt(pc.explained_variance_)
    ZB=pc.transform((B-mu)/sd)/np.sqrt(pc.explained_variance_)
    DAA=cdist(ZA,ZA); np.fill_diagonal(DAA,np.inf)
    DBA=cdist(ZB,ZA)
    qa,qb=rank(len(A)),rank(len(B))
    def local(D):
        idx=np.argpartition(D,k-1,axis=1)[:,:k]
        d=np.take_along_axis(D,idx,axis=1)
        w=1/np.maximum(d,.05)**2
        return np.sum(w*qa[idx],axis=1)/w.sum(1)
    ref=local(DAA); query=local(DBA)
    ref_trend=gaussian_filter1d(ref,max(1,.05*len(A)),mode='nearest')
    delta=query-np.interp(qb,qa,ref_trend)
    delta=gaussian_filter1d(delta,max(1,.10*len(B)),mode='nearest')
    newq=np.clip(qb+alpha*4*qb*(1-qb)*delta,0,1)
    return IsotonicRegression(increasing=True).fit_transform(qb,newq)

def neighbor_signals(A,B,k=10,dim=12):
    mu,sd=A.mean(0),A.std(0)
    pc=PCA(n_components=dim,svd_solver='full').fit((A-mu)/sd)
    ZA=pc.transform((A-mu)/sd)/np.sqrt(pc.explained_variance_)
    ZB=pc.transform((B-mu)/sd)/np.sqrt(pc.explained_variance_)
    DA=cdist(ZA,ZA);np.fill_diagonal(DA,np.inf)
    DB=cdist(ZB,ZA)
    qa=rank(len(A))
    def local(D):
        ix=np.argpartition(D,k-1,axis=1)[:,:k]
        weights=1/np.maximum(np.take_along_axis(D,ix,axis=1),.05)**2
        return (weights*qa[ix]).sum(1)/weights.sum(1)
    return local(DA),local(DB)

def warp(A,B,k=10,dim=12,kind='gaussian',strength=1.):
    ref,query=neighbor_signals(A,B,k,dim)
    qa,qb=rank(len(A)),rank(len(B))
    trend=gaussian_filter1d(ref,.05*len(A),mode='nearest')
    error=query-np.interp(qb,qa,trend)
    if kind=='bridge':
        # Gaussian approximation to train/query empirical rank displacement.
        # min(u,v)-u*v is the Brownian bridge covariance kernel.
        K=(np.minimum.outer(qb,qb)-np.outer(qb,qb))*(1/len(A)+1/len(B))*strength
        derivative=np.clip(np.interp(qb,qa,np.gradient(trend,qa)),.1,3.)
        sigma2=max(float(np.mean((ref-trend)**2)),1e-4)
        cov=K*derivative[:,None]*derivative[None,:]+np.eye(len(B))*sigma2
        correction=(K*derivative[None,:])@np.linalg.solve(cov,error)
    else:
        correction=strength*4*qb*(1-qb)*gaussian_filter1d(error,max(1,.1*len(B)),mode='nearest')
    return IsotonicRegression(increasing=True).fit_transform(qb,np.clip(qb+correction,0,1))

def all_candidates(A,t,B):
    out={}
    for dim in [8,12,16,20]:
        u=warp(A,B,dim=dim)
        out[f'gaussian_dim{dim}']=curve(t,u,.005)
    out['dimension_mean']=np.mean([out[f'gaussian_dim{d}'] for d in [8,12,16,20]],axis=0)
    for scale in [.5,1.,2.]:
        u=warp(A,B,kind='bridge',strength=scale)
        out[f'bridge_{scale}']=curve(t,u,.005)
    rng=np.random.default_rng(5100)
    bags=[]
    for _ in range(16):
        a=np.sort(rng.choice(len(A),int(.9*len(A)),replace=False))
        bags.append(curve(t[a],warp(A[a],B),.005))
    out['subbag16']=np.mean(bags,axis=0)
    out['subbag_half_full']=.5*(out['subbag16']+out['gaussian_dim12'])
    return out

def residual_candidates(A,t,B):
    qa,qb=rank(len(A)),warp(A,B)
    mu,sd=A.mean(0),A.std(0)
    Z=(A-mu)/sd;Q=(B-mu)/sd
    smooth_x=gaussian_filter1d(Z,.05*len(A),axis=0,mode='nearest')
    smooth_y=gaussian_filter1d(t,.05*len(A),mode='nearest')
    RX=Z-smooth_x
    RQ=Q-np.column_stack([np.interp(qb,qa,smooth_x[:,j]) for j in range(Z.shape[1])])
    base=curve(t,qb,.005)
    out={'none':base}
    for alpha in [10,100]:
        model=Ridge(alpha=alpha).fit(RX,t-smooth_y)
        residual=model.predict(RQ)
        for weight in [.5,1.]:out[f'ridge{alpha}_w{weight}']=base+weight*residual
    return out

In [ ]:
KEYS = ['gaussian_dim8', 'gaussian_dim12', 'dimension_mean', 'bridge_0.5', 'subbag16', 'ridge10_w0.5', 'ridge10_w1.0', 'ridge100_w0.5', 'ridge100_w1.0']

def candidates(A,t,B):
    all_p=all_candidates(A,t,B)
    all_p.update(residual_candidates(A,t,B))
    return {k:all_p[k] for k in KEYS}

def select_rule(A,t,seed):
    losses={k:0. for k in KEYS}
    for a,b in KFold(5,shuffle=True,random_state=seed).split(A):
        a,b=np.sort(a),np.sort(b)
        assert not set(a)&set(b)
        for key,p in candidates(A[a],t[a],A[b]).items():
            losses[key]+=np.sum((p-t[b])**2)
    return min(losses,key=losses.get),{k:float(np.sqrt(v/len(t))) for k,v in losses.items()}

## Nested validation

The 15 outer folds match the archived v10 predictions. For each fold, candidate selection uses only the outer training rows. All transforms, smoothing trends and residual models are fitted there. The final blend weight remains 50:50.

Optional v10 OOF is the prior nested local audit baseline, not a claim of bitwise identical Kaggle predictions. The baseline and candidate catalogue were developed on this same dataset; results remain development estimates.

In [ ]:
start = time.perf_counter()
folds = list(RepeatedKFold(n_splits=5, n_repeats=3, random_state=42).split(X))
rank_oof, quantile_oof = np.zeros((3, len(y))), np.zeros((3, len(y)))
fold_rows = []
for i, (a, b) in enumerate(folds):
    a, b = np.sort(a), np.sort(b)
    assert not set(a) & set(b)
    selected, inner_scores = select_rule(X[a], y[a], 5200 + i)
    predictions = candidates(X[a], y[a], X[b])
    rank_oof[i // 5, b] = predictions[selected]
    quantile_oof[i // 5, b] = curve(y[a], rank(len(b)))
    fold_rows.append({'fold': i, 'selected': selected, 'inner_rmse': inner_scores[selected]})
    print(f'Fold {i+1}/15: {selected}')

predictions = {'quantile': quantile_oof, 'nested_rank': rank_oof}
if baseline_oof is not None:
    predictions['v10'] = baseline_oof
    predictions['v11'] = .5 * (baseline_oof + rank_oof)
rows = []
for name, p in predictions.items():
    fs = [np.sqrt(np.mean((p[i//5, b] - y[b])**2)) for i, (_, b) in enumerate(folds)]
    reps = [np.sqrt(np.mean((pp-y)**2)) for pp in p]
    rows.append({'method': name, 'mean_fold_rmse': np.mean(fs), 'mean_repeat_rmse': np.mean(reps),
                 'repeat1': reps[0], 'repeat2': reps[1], 'repeat3': reps[2]})
scores = pd.DataFrame(rows)
print(scores.round(4).to_string(index=False))
print(f'CPU diagnostic elapsed: {time.perf_counter()-start:.1f} seconds')
scores.to_csv(OUT / 'validation.csv', index=False)
pd.DataFrame(fold_rows).to_csv(OUT / 'fold_selection.csv', index=False)
np.savez(OUT / 'validation_predictions.npz', **predictions)


## Export and provenance

Choose one candidate using five-fold CV on all training rows, then average its test prediction with the exact v10 input file. A manifest records source-file hashes, dependency versions, selected model, seed, and output hash. The export is rerun to verify byte-identical CSV output in this environment. No submission is uploaded automatically.

In [ ]:
selected, inner_scores = select_rule(X, y, 5299)
new_test = candidates(X, y, Xte)[selected]
final_test = .5 * (v10_test + new_test)
assert np.isfinite(final_test).all()
assert set(template.record_id) == set(te.record_id) and template.record_id.is_unique

def submission_frame(p):
    result = template[['record_id']].merge(
        pd.DataFrame({'record_id': te.record_id, 'prediction': p}),
        on='record_id', how='left', validate='one_to_one')
    assert len(result) == len(te) and np.isfinite(result.prediction).all()
    return result

result = submission_frame(final_test)
csv_text = result.to_csv(index=False)
output_path = OUT / 'submission_v11.csv'
output_path.write_text(csv_text)
repeat_selected, _ = select_rule(X, y, 5299)
repeat_new = candidates(X, y, Xte)[repeat_selected]
repeat_text = submission_frame(.5 * (v10_test + repeat_new)).to_csv(index=False)
assert selected == repeat_selected and csv_text == repeat_text

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

manifest = {
    'selected': selected, 'full_training_cv': inner_scores,
    'v10_weight': .5, 'new_model_weight': .5, 'selection_seed': 5299,
    'versions': VERSIONS, 'platform': platform.platform(),
    'input_sha256': {str(p): sha256_file(p) for p in [DATA_DIR/'train.csv', DATA_DIR/'test.csv',
                     DATA_DIR/'submission_template.csv', V10_PATH]},
    'submission_sha256': sha256_file(output_path),
    'same_environment_repeat_identical': True,
    'v10_public_reported_by_user': 2.1068098327, 'v11_public': None,
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))
(OUT / 'requirements-v11.txt').write_text('\n'.join(f'{k}=={v}' for k,v in VERSIONS.items() if k != 'python')+'\n')
changes = pd.DataFrame({'record_id': te.record_id, 'v10': v10_test,
                       'new_model': new_test, 'v11': final_test, 'change': final_test-v10_test})
changes.to_csv(OUT / 'prediction_changes.csv', index=False)
print('Selected:', selected)
print('Repeated export is byte-identical:', True)
print('RMS prediction change:', np.sqrt(np.mean((final_test-v10_test)**2)))
print('Submission SHA256:', manifest['submission_sha256'])
print(result.head().to_string(index=False))


The local nested audit gave mean fold RMSE 3.4008 for v10 and 3.3031 for v11. All three repeats improved. The paired bootstrap conditions on the stored predictions and does not cover all method-selection uncertainty. These validation gains do not guarantee a lower public score. CPU execution is sufficient; foundation models are not retrained.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(rank(len(te)), v10_test, label='v10')
axes[0].plot(rank(len(te)), new_test, label='rank + residual', alpha=.8)
axes[0].plot(rank(len(te)), final_test, label='v11')
axes[0].set(xlabel='Position within test file', ylabel='Prediction', title='Final candidate predictions')
axes[0].legend()
axes[1].plot(rank(len(te)), final_test-v10_test)
axes[1].axhline(0, color='black', lw=1)
axes[1].set(xlabel='Position within test file', ylabel='v11 minus v10', title='Changes do not use test labels')
fig.tight_layout()
fig.savefig(OUT / 'v11_predictions.png', dpi=150)
print('Saved plot:', OUT / 'v11_predictions.png')
